[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/15_cross_validation/03_nested_cv_and_protocol.ipynb)

# Bài 03 — Nested CV, rò rỉ trong CV và quy trình đánh giá chuẩn

**🎯 Mục tiêu**
- Thấy **thiên lệch lạc quan** khi báo cáo điểm CV tốt nhất sau khi tune nhiều cấu hình (selection bias).
- **Nested CV**: vòng ngoài ước lượng hiệu năng, vòng trong chọn hyperparameter.
- Rò rỉ kinh điển: **chọn feature trên toàn bộ dữ liệu** rồi mới CV (Ambroise & McLachlan, 2002) — và cách Pipeline ngăn nó.
- Quy trình đánh giá chuẩn từ đầu đến cuối.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/15_cross_validation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, KFold, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)

## 1. Rò rỉ kinh điển: chọn feature trước khi CV

Dữ liệu **thuần nhiễu**: 100 mẫu, 5 000 feature ngẫu nhiên, nhãn ngẫu nhiên — **không có gì để học**, accuracy thật = 50%.

In [ ]:
X_noise = rng.normal(size=(100, 5000))
y_noise = rng.integers(0, 2, 100)

# ❌ SAI: chọn 20 feature "tốt nhất" trên TOÀN BỘ dữ liệu, rồi mới cross-validate
selected = SelectKBest(f_classif, k=20).fit(X_noise, y_noise).get_support()
wrong = cross_val_score(LogisticRegression(max_iter=2000), X_noise[:, selected], y_noise, cv=StratifiedKFold(5, shuffle=True, random_state=0)).mean()

# ✅ ĐÚNG: chọn feature BÊN TRONG mỗi fold (Pipeline)
pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=2000))
right = cross_val_score(pipe, X_noise, y_noise, cv=StratifiedKFold(5, shuffle=True, random_state=0)).mean()
print(f"chọn feature trước CV  : accuracy = {wrong:.3f}   ← ảo tưởng")
print(f"chọn feature trong CV  : accuracy = {right:.3f}   ← trung thực (≈ 0.5)")

Với 5 000 feature ngẫu nhiên, luôn có vài feature **tình cờ** tương quan với nhãn trên 100 mẫu này — chọn chúng bằng cả dữ liệu validation nghĩa là validation đã "rò" vào quá trình xây model. **Mọi** bước học từ dữ liệu (scaling, impute, encoding theo target, chọn feature, PCA, oversampling...) phải nằm **trong** Pipeline để được fit lại trên train của từng fold.

## 2. Selection bias khi tune hyperparameter

Dữ liệu nhiễu nhỏ hơn (60 mẫu, 20 feature, nhãn ngẫu nhiên). Thử 30 cấu hình KNN (`n_neighbors`, `weights`) rồi **báo cáo điểm CV của cấu hình tốt nhất**. Lặp lại trên **10 bộ dữ liệu nhiễu** khác nhau để thấy xu hướng, không phải may rủi:

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_grid = {"kneighborsclassifier__n_neighbors": list(range(1, 30, 2)),
            "kneighborsclassifier__weights": ["uniform", "distance"]}
best_scores, nested_scores = [], []
for seed in range(10):
    r = np.random.default_rng(100 + seed)
    Xs, ys = r.normal(size=(60, 20)), r.integers(0, 2, 60)
    search = GridSearchCV(make_pipeline(StandardScaler(), KNeighborsClassifier()), knn_grid,
                          cv=StratifiedKFold(5, shuffle=True, random_state=1))
    best_scores.append(search.fit(Xs, ys).best_score_)
    # nested CV: vòng ngoài đánh giá CẢ quy trình "GridSearchCV" (mục 3)
    nested_scores.append(cross_val_score(search, Xs, ys, cv=StratifiedKFold(5, shuffle=True, random_state=2)).mean())
print(f"điểm CV tốt nhất (30 cấu hình), trung bình 10 bộ dữ liệu: {np.mean(best_scores):.3f}   ← lạc quan")
print(f"nested CV,                      trung bình 10 bộ dữ liệu: {np.mean(nested_scores):.3f}   ← trung thực (≈ 0.5)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
for b, n in zip(best_scores, nested_scores):
    ax.plot([0, 1], [b, n], color="0.75", lw=1)
ax.scatter(np.zeros(10), best_scores, color="C3", zorder=3, label="điểm CV của cấu hình tốt nhất")
ax.scatter(np.ones(10), nested_scores, color="C0", zorder=3, label="nested CV")
ax.axhline(0.5, color="k", ls="--", label="sự thật: 0.5 (nhãn ngẫu nhiên)")
ax.set_xticks([0, 1], ["chọn rồi báo cáo luôn", "nested CV"])
ax.set_xlim(-0.4, 1.4)
ax.set_ylabel("accuracy ước lượng")
ax.set_title("10 bộ dữ liệu thuần nhiễu — mỗi đường xám là một bộ")
ax.legend(loc="upper right", fontsize=9)
plt.show()

Điểm "tốt nhất" là **max** của 30 ước lượng nhiễu → luôn lớn hơn 0.5 dù không có tín hiệu. Thử càng nhiều cấu hình, dữ liệu càng ít, thiên lệch càng lớn. Con số này hợp lệ để **chọn** cấu hình, không hợp lệ để **báo cáo** hiệu năng.

## 3. Nested cross-validation

- **Vòng trong** (inner CV): trên train của mỗi fold ngoài, `GridSearchCV` chọn hyperparameter.
- **Vòng ngoài** (outer CV): đánh giá **toàn bộ quy trình** (bao gồm cả bước tune) trên fold chưa từng thấy.

Trong scikit-learn chỉ cần một dòng: `cross_val_score(GridSearchCV(...), X, y, cv=outer)` — như ở cell trên.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.4), gridspec_kw={"width_ratios": [5, 4]})
for ax, n, color, name in [(axes[0], 5, "#d62728", "ngoài"), (axes[1], 4, "#ff7f0e", "trong")]:
    for i in range(n):
        for j in range(n):
            ax.add_patch(plt.Rectangle((j, n - 1 - i), 0.95, 0.8, color=color if i == j else "#1f77b4", alpha=0.85 if i == j else 0.35))
        ax.text(-0.15, n - 1 - i + 0.4, f"fold {name} {i + 1}", ha="right", va="center", fontsize=9)
    ax.set_xlim(-1.7, n)
    ax.set_ylim(-0.2, n)
    ax.axis("off")
axes[0].set_title("Vòng ngoài: đánh giá CẢ quy trình (đỏ = phần test)")
axes[1].set_title("Vòng trong — chỉ trên phần xanh của MỘT fold ngoài:\nchọn hyperparameter (cam = validation)")
plt.tight_layout()
plt.show()

### Nested CV from scratch (để thấy rõ cấu trúc)

In [ ]:
from ml_course.datasets import make_customer_churn

churn = make_customer_churn(n=2000, seed=6)
Xc = pd.get_dummies(churn.drop(columns=["customer_id", "churn"]), dtype=float).to_numpy()
yc = churn["churn"].to_numpy()
Cs = [0.001, 0.01, 0.1, 1, 10]

outer_scores, chosen = [], []
for train, test in StratifiedKFold(5, shuffle=True, random_state=0).split(Xc, yc):
    # vòng trong: chỉ dùng dữ liệu train của fold ngoài
    inner_means = [cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000)),
                                   Xc[train], yc[train], cv=StratifiedKFold(4, shuffle=True, random_state=0), scoring="roc_auc").mean()
                   for C in Cs]
    best_C = Cs[int(np.argmax(inner_means))]
    model = make_pipeline(StandardScaler(), LogisticRegression(C=best_C, max_iter=3000)).fit(Xc[train], yc[train])

    outer_scores.append(roc_auc_score(yc[test], model.predict_proba(Xc[test])[:, 1]))
    chosen.append(best_C)
print("C được chọn ở mỗi fold ngoài:", chosen)
print(f"nested CV ROC-AUC = {np.mean(outer_scores):.4f} ± {np.std(outer_scores):.4f}")

C được chọn có thể **khác nhau** giữa các fold — nested CV đánh giá **quy trình** "tune C bằng CV rồi train", không phải một giá trị C cụ thể. Model cuối cùng: chạy lại quy trình (GridSearchCV) trên **toàn bộ** dữ liệu.

## 4. Quy trình đánh giá chuẩn

```text
1. Tách TEST (hold-out cuối cùng) ngay từ đầu — theo cách mô phỏng triển khai (thời gian / nhóm / ngẫu nhiên)
   → không nhìn vào nó cho tới bước 6.
2. Trên phần còn lại (DEV): EDA, feature engineering, chọn model — tất cả trong Pipeline.
3. Chọn hyperparameter bằng CV trên DEV (chiến lược CV đúng: stratified / group / time).
4. (Nếu cần ước lượng không thiên lệch của cả quy trình mà dữ liệu ít) → nested CV trên DEV.
5. Chọn ngưỡng / calibration trên dự đoán out-of-fold hoặc một tập validation riêng.
6. Train lại model cuối trên toàn bộ DEV → đánh giá MỘT LẦN trên TEST, báo cáo kèm khoảng tin cậy.
7. (Tuỳ chọn) Train lại trên DEV + TEST trước khi triển khai — nhưng con số báo cáo vẫn là của bước 6.
```

In [ ]:
X_dev, X_test, y_dev, y_test = train_test_split(Xc, yc, test_size=0.2, random_state=0, stratify=yc)
search = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)), {"logisticregression__C": Cs},
                      cv=StratifiedKFold(5, shuffle=True, random_state=0), scoring="roc_auc").fit(X_dev, y_dev)

test_auc = roc_auc_score(y_test, search.predict_proba(X_test)[:, 1])
print(f"CV tốt nhất trên DEV = {search.best_score_:.4f} (C = {search.best_params_['logisticregression__C']}) | TEST (một lần) = {test_auc:.4f}")

### Các lỗi thường gặp

| Lỗi | Hậu quả |
|---|---|
| Scale/impute/chọn feature trên toàn bộ dữ liệu trước khi chia | rò rỉ, lạc quan |
| Báo cáo điểm CV tốt nhất sau khi tune | selection bias |
| Dùng test nhiều lần để chọn model | test thành validation, mất ước lượng trung thực |
| KFold ngẫu nhiên cho dữ liệu nhóm / thời gian | lạc quan nghiêm trọng (bài 02) |
| Oversampling (SMOTE) trước khi chia | mẫu tổng hợp từ validation lọt vào train (chương 19) |
| Target encoding trên toàn bộ dữ liệu | rò rỉ nhãn (chương 17) |

## 🧠 Kiểm tra nhanh

1. Vì sao điểm CV tốt nhất của GridSearchCV là ước lượng lạc quan?
2. Nested CV ước lượng hiệu năng của cái gì — một model cụ thể hay một quy trình?
3. Bạn đã xem kết quả test, thấy kém, quay lại đổi feature rồi test lại. Kết quả test mới còn trung thực không?
4. Liệt kê 4 bước tiền xử lý phải nằm trong Pipeline.

## 📌 Tóm tắt

- Mọi bước học từ dữ liệu phải nằm trong Pipeline để fit lại trong từng fold.
- Điểm CV tốt nhất sau tune bị thiên lệch lạc quan; nested CV cho ước lượng trung thực của cả quy trình.
- Test chỉ dùng một lần, ở cuối.

## 📚 Tài liệu tham khảo

- Ambroise & McLachlan (2002) — *Selection bias in gene extraction on the basis of microarray gene-expression data*
- Cawley & Talbot (2010) — *On Over-fitting in Model Selection and Subsequent Selection Bias in Performance Evaluation*
- Varma & Simon (2006) — *Bias in error estimation when using cross-validation for model selection*